In [0]:
def merge(
    df_source,
    target_table,
    merge_key,
    timestamp_col="ingestion_timestamp",
    transforms=None,
    exclude_update_cols=None
):
    if df_source.isEmpty():
        return
    
    source_view = "updates"
    df_source.createOrReplaceTempView(source_view)
    
    columns = df_source.columns
    
    exclude_update_cols = exclude_update_cols or []
    
    # 🔹 ON dinámico
    if isinstance(merge_key, list):
        on_clause = " AND ".join([f"t.{col} = s.{col}" for col in merge_key])
    else:
        on_clause = f"t.{merge_key} = s.{merge_key}"
    
    # 🔹 transformaciones
    def apply_transform(col):
        if transforms and col in transforms:
            return transforms[col]
        return f"s.{col}"
    
    # 🔹 UPDATE dinámico
    update_set = ",\n        ".join([
        f"t.{col} = {apply_transform(col)}"
        for col in columns
        if col not in exclude_update_cols and col != timestamp_col
    ])
    
    # 🔹 INSERT dinámico
    insert_cols = ",\n        ".join(columns)
    insert_values = ",\n        ".join([
        apply_transform(col) for col in columns
    ])
    
    merge_sql = f"""
    MERGE INTO {target_table} t
    USING {source_view} s
    ON {on_clause}
    
    WHEN MATCHED AND s.{timestamp_col} > t.{timestamp_col} THEN
    UPDATE SET
        {update_set}
    
    WHEN NOT MATCHED THEN
    INSERT (
        {insert_cols}
    )
    VALUES (
        {insert_values}
    )
    """
    
    spark.sql(merge_sql)

In [0]:
def mergex(
    df_source,
    target_table,
    merge_key,
    timestamp_col="ingestion_timestamp",
    transforms=None,  # columnas con funciones (ej: INITCAP)
    exclude_update_cols=None
):
    #if df_source.count() == 0:
    if df_source.isEmpty():
        return
    
    source_view = "updates"
    df_source.createOrReplaceTempView(source_view)
    
    columns = df_source.columns
    
    # excluir columnas si es necesario
    exclude_update_cols = exclude_update_cols or []
    
    # 🔹 aplicar transformaciones
    def apply_transform(col):
        if transforms and col in transforms:
            return transforms[col]
        return f"s.{col}"
    
    # 🔹 UPDATE dinámico
    update_set = ",\n        ".join([
        f"t.{col} = {apply_transform(col)}"
        for col in columns
        if col != merge_key and col not in exclude_update_cols
    ])
    
    # 🔹 INSERT dinámico
    insert_cols = ",\n        ".join(columns)
    insert_values = ",\n        ".join([
        apply_transform(col) for col in columns
    ])
    
    merge_sql = f"""
    MERGE INTO {target_table} t
    USING {source_view} s
    ON t.{merge_key} = s.{merge_key}
    
    WHEN MATCHED AND s.{timestamp_col} > t.{timestamp_col} THEN
    UPDATE SET
        {update_set}
    
    WHEN NOT MATCHED THEN
    INSERT (
        {insert_cols}
    )
    VALUES (
        {insert_values}
    )
    """
    
    spark.sql(merge_sql)